# 04 — Common Crawl Extraction (Système Big Data — Bonus)

**Task 1.7b** — Sicurre Data Pipeline  
**Compétence C1** — Extraire des données à partir de sources hétérogènes (système big data)

---

## Objectif

Common Crawl archive **des milliards de pages web** (~250 To/mois). Ce n'est pas un corpus
d'emails, mais on peut en extraire :

- **Pages de phishing francophones** : fausses pages bancaires, formulaires de vol de credentials
- **Pages d'arnaques FR** : faux colis, faux remboursements, faux support technique
- **Texte légitime FR** (ham) : pages de banques réelles, sites gouvernementaux

### Stratégie : Hyper-filtrage

1. Interroger l'**Index API** de Common Crawl (pas besoin d'AWS/Athena)
2. Filtrer par domaines associés au phishing FR (`.fr`, termes bancaires, patterns d'arnaque)
3. Télécharger uniquement les **WARC records** correspondants (~Ko, pas To)
4. Extraire le texte, valider la qualité **avant** tout nettoyage
5. Exporter un petit corpus validé vers `data/raw/common_crawl/`

### ⚠️ Ce notebook est exploratoire

On vérifie d'abord ce qu'on peut récupérer et sa qualité. Aucun pipeline de
nettoyage n'est construit ici — c'est du **data quality assessment**.

**Deps :** `httpx`, `warcio`, `beautifulsoup4`, `langdetect`, `pandas` — gérées par `uv`.

In [ ]:
# ── Imports & Constants ──────────────────────────────────────────────
from __future__ import annotations

import hashlib
import io
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import httpx
import pandas as pd
from bs4 import BeautifulSoup
from langdetect import detect, LangDetectException
from warcio.archiveiterator import ArchiveIterator

# ── Configuration ────────────────────────────────────────────────────
# Common Crawl Index API (latest crawl)
CC_INDEX_URL: str = "https://index.commoncrawl.org/CC-MAIN-2025-08-index"  # Feb 2025 crawl

OUTPUT_DIR: Path = Path("../../data/raw/bigdata/common_crawl")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Limits — we're exploring, not bulk-downloading
MAX_RESULTS_PER_QUERY: int = 50
MAX_WARC_DOWNLOADS: int = 30       # Cap WARC fetches to be respectful
MIN_TEXT_LENGTH: int = 100          # Chars — skip tiny pages
MAX_TEXT_LENGTH: int = 10_000       # Chars — skip huge pages
REQUEST_TIMEOUT: int = 30           # Seconds

print(f"CC Index    : {CC_INDEX_URL}")
print(f"Output dir  : {OUTPUT_DIR.resolve()}")
print(f"Max results : {MAX_RESULTS_PER_QUERY}/query")
print(f"Max WARCs   : {MAX_WARC_DOWNLOADS}")

## 1. Query Common Crawl Index API

The [CC Index API](https://index.commoncrawl.org/) lets us search URLs by pattern
without downloading terabytes of data. We query for:

- **Phishing patterns:** domains mimicking French banks (bnp, credit-agricole, la-banque-postale, etc.)
- **Scam patterns:** fake parcel delivery, fake government sites
- **Legitimate FR:** real bank/gov sites (for ham comparison)

Each result gives us the WARC file location so we can fetch **just that one page**.

In [ ]:
# ── CC Index Query Helper ────────────────────────────────────────────
def query_cc_index(
    url_pattern: str,
    *,
    max_results: int = MAX_RESULTS_PER_QUERY,
    filters: list[str] | None = None,
) -> list[dict]:
    """Query the Common Crawl Index API for URLs matching a pattern.
    
    Args:
        url_pattern: URL pattern to search (supports wildcards via CC syntax).
        max_results: Maximum number of results to return.
        filters: Optional CC Index filters (e.g., ['=status:200']).
    
    Returns:
        List of index records with WARC location metadata.
    """
    params = {
        "url": url_pattern,
        "output": "json",
        "limit": str(max_results),
    }
    if filters:
        params["filter"] = "&".join(filters)

    try:
        resp = httpx.get(
            CC_INDEX_URL,
            params=params,
            timeout=REQUEST_TIMEOUT,
            follow_redirects=True,
        )
        resp.raise_for_status()

        # CC Index returns one JSON object per line (NDJSON)
        records = [
            json.loads(line)
            for line in resp.text.strip().split("\n")
            if line.strip()
        ]
        return records

    except httpx.HTTPStatusError as e:
        print(f"⚠️  HTTP {e.response.status_code} for pattern '{url_pattern}'")
        return []
    except Exception as e:
        print(f"⚠️  Error querying CC Index: {e}")
        return []


# Quick connectivity test
if test:= query_cc_index("fr.wikipedia.org/wiki/Hameçonnage", max_results=1):
    print(f"✅ CC Index API is reachable — got {len(test)} result(s)")
    print(f"   Sample URL: {test[0].get('url', 'N/A')}")
else:
    print("⚠️  CC Index returned no results. Try a different crawl ID or check connectivity.")

## 2. Define Search Queries (Hyper-filtered)

We define **targeted queries** for two categories:

| Category | Goal | URL patterns |
|----------|------|--------------|
| **Phishing-like** | Fake bank/gov pages, scam sites | Typosquatting domains, suspicious TLDs |
| **Legitimate FR** | Real French banks, gov, services | Known-good `.fr` / `.gouv.fr` domains |

This gives us **both classes** (phishing + ham) from the same big data source.

In [ ]:
# ── Search Queries ────────────────────────────────────────────────────
# Phishing-like: typosquatting, suspicious TLDs, known scam patterns
# These are URL patterns that LOOK like phishing (common typosquat targets)
PHISHING_QUERIES: list[dict[str, str]] = [
    # Fake banking pages (typosquatting patterns)
    {"pattern": "*.bnp-paribas-secure.*", "label": "phishing_bank"},
    {"pattern": "*.credit-agricole-login.*", "label": "phishing_bank"},
    {"pattern": "*.la-banque-postale-verification.*", "label": "phishing_bank"},
    {"pattern": "*.societe-generale-secure.*", "label": "phishing_bank"},
    # Generic FR phishing patterns
    {"pattern": "*.connexion-securisee.fr*", "label": "phishing_generic"},
    {"pattern": "*.verification-compte.*", "label": "phishing_generic"},
    {"pattern": "*.mise-a-jour-securite.*", "label": "phishing_generic"},
    # Fake parcel / colis
    {"pattern": "*.colissimo-suivi.*", "label": "phishing_colis"},
    {"pattern": "*.chronopost-livraison.*", "label": "phishing_colis"},
    {"pattern": "*.la-poste-colis.*", "label": "phishing_colis"},
    # Fake government
    {"pattern": "*.ameli-remboursement.*", "label": "phishing_gov"},
    {"pattern": "*.impots-gouv-fr.*", "label": "phishing_gov"},
    {"pattern": "*.caf-allocation.*", "label": "phishing_gov"},
]

# Legitimate FR sites (for ham / comparison)
LEGITIMATE_QUERIES: list[dict[str, str]] = [
    {"pattern": "www.mabanque.bnpparibas/*", "label": "legit_bank"},
    {"pattern": "www.credit-agricole.fr/*", "label": "legit_bank"},
    {"pattern": "www.labanquepostale.fr/*", "label": "legit_bank"},
    {"pattern": "www.ameli.fr/*", "label": "legit_gov"},
    {"pattern": "www.impots.gouv.fr/*", "label": "legit_gov"},
    {"pattern": "www.caf.fr/*", "label": "legit_gov"},
    {"pattern": "www.colissimo.fr/*", "label": "legit_colis"},
]

print(f"Phishing queries  : {len(PHISHING_QUERIES)}")
print(f"Legitimate queries : {len(LEGITIMATE_QUERIES)}")
print(f"Total queries      : {len(PHISHING_QUERIES) + len(LEGITIMATE_QUERIES)}")

In [ ]:
# ── Execute All Queries ──────────────────────────────────────────────
all_queries = [
    *[(q, "phishing") for q in PHISHING_QUERIES],
    *[(q, "legitimate") for q in LEGITIMATE_QUERIES],
]

all_records: list[dict] = []

for query_def, category in all_queries:
    pattern = query_def["pattern"]
    label = query_def["label"]
    
    records = query_cc_index(pattern, max_results=MAX_RESULTS_PER_QUERY)
    
    for rec in records:
        rec["_category"] = category
        rec["_label"] = label
        rec["_query"] = pattern
    
    all_records.extend(records)
    
    status = f"{len(records):3d} results" if records else "  0 results"
    print(f"  [{category:10s}] {pattern:45s} → {status}")
    
    # Be polite — 0.5s between requests
    time.sleep(0.5)

print(f"\n📊 Total index records: {len(all_records)}")

## 3. Inspect Index Results (Before Downloading)

**Quality gate #1:** Before downloading any WARC data, inspect what the index returned.
Check URL patterns, HTTP status codes, MIME types, and timestamps.

In [ ]:
# ── Convert to DataFrame for analysis ────────────────────────────────
if not all_records:
    print("⚠️  No records found. This is expected — phishing domains are short-lived.")
    print("   Common Crawl may not have captured them. Skipping remaining cells.")
    print("   This is itself a valuable finding for the rapport.")
    df_index = pd.DataFrame()  # Empty DF so downstream cells don't crash
else:
    df_index = pd.DataFrame(all_records)
    
    print(f"── Index Results Summary ──")
    print(f"Total records     : {len(df_index):,}")
    print(f"Unique URLs       : {df_index['url'].nunique():,}")
    print(f"\n── By Category ──")
    print(df_index["_category"].value_counts().to_string())
    print(f"\n── By Label ──")
    print(df_index["_label"].value_counts().to_string())
    
    # HTTP status distribution
    if "status" in df_index.columns:
        print(f"\n── HTTP Status ──")
        print(df_index["status"].value_counts().head(5).to_string())
    
    # MIME types
    if "mime" in df_index.columns:
        print(f"\n── MIME Types ──")
        print(df_index["mime"].value_counts().head(5).to_string())

In [ ]:
# ── Preview URLs ─────────────────────────────────────────────────────
if not df_index.empty:
    print("── Sample URLs by Category ──\n")
    for cat in df_index["_category"].unique():
        subset = df_index[df_index["_category"] == cat]
        print(f"\n🔹 {cat.upper()} ({len(subset)} records):")
        for url in subset["url"].head(5):
            print(f"   {url[:100]}")

## 4. Download WARC Records & Extract Text

For each index hit, Common Crawl gives us `filename`, `offset`, and `length`.
We use HTTP Range requests to fetch **only** the relevant bytes from the WARC — 
not the whole multi-GB file.

**Quality gate #2:** We validate each page as we extract it.

In [ ]:
# ── WARC Fetch & Text Extraction ─────────────────────────────────────
CC_WARC_BASE: str = "https://data.commoncrawl.org/"


def fetch_warc_record(filename: str, offset: int, length: int) -> bytes | None:
    """Fetch a single WARC record using an HTTP Range request.
    
    Args:
        filename: WARC file path in CC S3 bucket.
        offset: Byte offset of the record.
        length: Byte length of the record.
    
    Returns:
        Raw WARC record bytes, or None on failure.
    """
    end = offset + length - 1
    url = f"{CC_WARC_BASE}{filename}"
    headers = {"Range": f"bytes={offset}-{end}"}
    
    try:
        resp = httpx.get(url, headers=headers, timeout=REQUEST_TIMEOUT, follow_redirects=True)
        if resp.status_code in (200, 206):
            return resp.content
        else:
            return None
    except Exception:
        return None


def extract_text_from_html(html: str) -> str:
    """Extract visible text from HTML, stripping scripts/styles."""
    soup = BeautifulSoup(html, "html.parser")
    
    # Remove non-visible elements
    for tag in soup.find_all(["script", "style", "meta", "link", "noscript"]):
        tag.decompose()
    
    text = soup.get_text(separator=" ", strip=True)
    
    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


def detect_language_safe(text: str) -> str:
    """Detect language, return 'unknown' on failure."""
    try:
        return detect(text[:1000])  # langdetect only needs first ~1000 chars
    except LangDetectException:
        return "unknown"


print("✅ Helper functions defined")

In [ ]:
# ── Download & Extract (with quality validation) ─────────────────────
extracted_pages: list[dict] = []
download_errors: int = 0
skipped_short: int = 0
skipped_lang: int = 0

if not df_index.empty:
    # Filter to HTML pages with status 200 only
    df_download = df_index.copy()
    if "status" in df_download.columns:
        df_download = df_download[df_download["status"].astype(str) == "200"]
    if "mime" in df_download.columns:
        df_download = df_download[df_download["mime"].str.contains("html", case=False, na=False)]
    
    # Cap downloads
    df_download = df_download.head(MAX_WARC_DOWNLOADS)
    
    print(f"Downloading {len(df_download)} WARC records...\n")
    
    for i, (_, row) in enumerate(df_download.iterrows()):
        # Fetch WARC record
        warc_data = fetch_warc_record(
            filename=row["filename"],
            offset=int(row["offset"]),
            length=int(row["length"]),
        )
        
        if warc_data is None:
            download_errors += 1
            continue
        
        # Parse WARC record
        try:
            stream = io.BytesIO(warc_data)
            for record in ArchiveIterator(stream):
                if record.rec_type == "response":
                    html = record.content_stream().read().decode("utf-8", errors="replace")
                    text = extract_text_from_html(html)
                    
                    # ── Quality Gate: Length ──
                    if len(text) < MIN_TEXT_LENGTH:
                        skipped_short += 1
                        continue
                    if len(text) > MAX_TEXT_LENGTH:
                        text = text[:MAX_TEXT_LENGTH]  # Truncate, don't skip
                    
                    # ── Quality Gate: Language ──
                    lang = detect_language_safe(text)
                    
                    extracted_pages.append({
                        "url": row["url"],
                        "text": text,
                        "text_length": len(text),
                        "language": lang,
                        "category": row["_category"],
                        "label": row["_label"],
                        "query": row["_query"],
                        "content_hash": hashlib.sha256(text.encode()).hexdigest()[:16],
                    })
        except Exception as e:
            download_errors += 1
            continue
        
        # Progress
        if (i + 1) % 10 == 0:
            print(f"  [{i+1}/{len(df_download)}] extracted: {len(extracted_pages)}, errors: {download_errors}")
        
        time.sleep(0.3)  # Rate limit
    
    print(f"\n── Download Summary ──")
    print(f"   Downloaded    : {len(df_download)}")
    print(f"   Extracted     : {len(extracted_pages)}")
    print(f"   Errors        : {download_errors}")
    print(f"   Skipped short : {skipped_short}")
else:
    print("⏭️  No index records to download.")

## 5. Data Quality Assessment

**This is the core of this notebook.** Before any cleaning pipeline, we validate:

1. **Language distribution** — how much is actually French?
2. **Text length distribution** — is the content meaningful?
3. **Deduplication** — how many unique pages?
4. **Content relevance** — does the text look like phishing / banking content?
5. **Signal-to-noise ratio** — what % is usable for our corpus?

In [ ]:
# ── Quality Assessment ────────────────────────────────────────────────
if extracted_pages:
    df_pages = pd.DataFrame(extracted_pages)
    
    print(f"═══════════════════════════════════════════")
    print(f"  DATA QUALITY REPORT — Common Crawl")
    print(f"═══════════════════════════════════════════")
    
    # ── 5a. Overview ──
    print(f"\n── Overview ──")
    print(f"   Total pages extracted : {len(df_pages)}")
    print(f"   Unique content hashes : {df_pages['content_hash'].nunique()}")
    dedup_rate = 1 - df_pages['content_hash'].nunique() / max(len(df_pages), 1)
    print(f"   Duplicate rate        : {dedup_rate:.1%}")
    
    # ── 5b. Language Distribution ──
    print(f"\n── Language Distribution ──")
    lang_counts = df_pages["language"].value_counts()
    for lang, count in lang_counts.items():
        pct = count / len(df_pages) * 100
        marker = " ← target" if lang == "fr" else ""
        print(f"   {lang:8s}: {count:4d} ({pct:5.1f}%){marker}")
    
    fr_count = lang_counts.get("fr", 0)
    fr_pct = fr_count / max(len(df_pages), 1) * 100
    print(f"\n   🇫🇷 French content: {fr_count} pages ({fr_pct:.1f}%)")
    
    # ── 5c. Text Length Stats ──
    print(f"\n── Text Length (chars) ──")
    print(f"   Mean   : {df_pages['text_length'].mean():.0f}")
    print(f"   Median : {df_pages['text_length'].median():.0f}")
    print(f"   Min    : {df_pages['text_length'].min()}")
    print(f"   Max    : {df_pages['text_length'].max():,}")
    
    # ── 5d. By Category ──
    print(f"\n── By Category ──")
    print(df_pages.groupby("category").agg(
        count=("text", "count"),
        fr_count=("language", lambda x: (x == "fr").sum()),
        avg_length=("text_length", "mean"),
    ).to_string())
    
else:
    df_pages = pd.DataFrame()
    print("⚠️  No pages extracted. Quality assessment skipped.")
    print("   This is a valid finding: Common Crawl has low coverage of phishing pages.")
    print("   Phishing sites are ephemeral — they're taken down before crawlers reach them.")

In [ ]:
# ── 5e. Content Relevance Spot-Check ─────────────────────────────────
# Manually inspect a few samples to assess relevance
if not df_pages.empty:
    print("── Content Spot-Check (first 200 chars per page) ──\n")
    
    # Show French pages first (most valuable)
    fr_pages = df_pages[df_pages["language"] == "fr"]
    sample = fr_pages.head(5) if len(fr_pages) >= 5 else df_pages.head(5)
    
    for _, row in sample.iterrows():
        print(f"🔹 [{row['category']}] [{row['language']}] {row['url'][:80]}")
        print(f"   {row['text'][:200]}")
        print()

In [ ]:
# ── 5f. Phishing Keyword Detection ───────────────────────────────────
# Check if extracted text contains phishing-related vocabulary
PHISHING_KEYWORDS_FR: list[str] = [
    "mot de passe", "identifiant", "connexion", "vérification",
    "sécurité", "compte bloqué", "urgent", "confirmer",
    "cliquez ici", "mettre à jour", "remboursement",
    "carte bancaire", "numéro de carte", "code secret",
    "livraison", "colis", "frais de douane",
]

if not df_pages.empty:
    def count_phishing_keywords(text: str) -> int:
        """Count phishing-related French keywords in text."""
        text_lower = text.lower()
        return sum(1 for kw in PHISHING_KEYWORDS_FR if kw in text_lower)
    
    df_pages["keyword_hits"] = df_pages["text"].apply(count_phishing_keywords)
    
    print("── Phishing Keyword Analysis ──")
    print(f"   Pages with ≥1 keyword  : {(df_pages['keyword_hits'] >= 1).sum()}")
    print(f"   Pages with ≥3 keywords : {(df_pages['keyword_hits'] >= 3).sum()}")
    print(f"   Avg keywords/page      : {df_pages['keyword_hits'].mean():.1f}")
    print(f"\n── Keyword Hits by Category ──")
    print(df_pages.groupby("category")["keyword_hits"].agg(["mean", "max", "sum"]).to_string())

## 6. Signal-to-Noise Verdict

Based on the quality assessment, determine what's usable.

In [ ]:
# ── Signal-to-Noise Summary ──────────────────────────────────────────
if not df_pages.empty:
    # Define "usable" — French, non-duplicate, with some keyword relevance
    df_usable = df_pages[
        (df_pages["language"] == "fr") &
        (df_pages["text_length"] >= MIN_TEXT_LENGTH)
    ].drop_duplicates(subset="content_hash")
    
    total = len(df_pages)
    usable = len(df_usable)
    ratio = usable / max(total, 1) * 100
    
    print(f"═══════════════════════════════════════════")
    print(f"  SIGNAL-TO-NOISE VERDICT")
    print(f"═══════════════════════════════════════════")
    print(f"   Total extracted     : {total}")
    print(f"   Usable (FR, dedup)  : {usable}")
    print(f"   Signal-to-Noise     : {ratio:.1f}%")
    
    if ratio >= 50:
        print(f"\n   ✅ Good signal — worth including in the pipeline")
    elif ratio >= 20:
        print(f"\n   ⚠️  Moderate signal — include with careful filtering")
    else:
        print(f"\n   ❌ Low signal — Common Crawl adds little value for this use case")
        print(f"      This is expected: phishing pages are ephemeral and rarely crawled.")
        print(f"      Document this finding in the rapport — it shows rigorous evaluation.")
else:
    print("No data to evaluate. Common Crawl returned 0 usable records.")
    print("This is itself a valid finding for the certification report.")

## 7. Export Validated Data

Export whatever passed quality gates to `data/raw/common_crawl/`.
Even a small validated set demonstrates the competency.

In [ ]:
# ── Export ────────────────────────────────────────────────────────────
if not df_pages.empty:
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d")
    
    # Export ALL extracted pages (with quality metadata)
    full_path = OUTPUT_DIR / f"common_crawl_all_{len(df_pages)}_{timestamp}.csv"
    df_pages.to_csv(full_path, index=False, encoding="utf-8")
    print(f"✅ Full export    : {full_path} ({len(df_pages)} rows)")
    
    # Export only usable FR subset
    if 'df_usable' in dir() and not df_usable.empty:
        usable_path = OUTPUT_DIR / f"common_crawl_fr_usable_{len(df_usable)}_{timestamp}.csv"
        df_usable.to_csv(usable_path, index=False, encoding="utf-8")
        print(f"✅ Usable FR only : {usable_path} ({len(df_usable)} rows)")
    
    # Export quality report as JSON
    report = {
        "extraction_date": timestamp,
        "cc_index": CC_INDEX_URL,
        "total_queries": len(PHISHING_QUERIES) + len(LEGITIMATE_QUERIES),
        "total_index_hits": len(df_index) if not df_index.empty else 0,
        "total_downloaded": MAX_WARC_DOWNLOADS,
        "total_extracted": len(df_pages),
        "download_errors": download_errors,
        "usable_french": len(df_usable) if 'df_usable' in dir() else 0,
        "language_distribution": df_pages["language"].value_counts().to_dict(),
        "category_distribution": df_pages["category"].value_counts().to_dict(),
    }
    report_path = OUTPUT_DIR / f"quality_report_{timestamp}.json"
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    print(f"✅ Quality report : {report_path}")
    
else:
    print("ℹ️  Nothing to export — 0 pages extracted.")
    print("   Create a note in your rapport documenting the negative result.")

## Résumé

| Métrique | Valeur |
|----------|--------|
| Source | Common Crawl (250+ To/mois, système big data) |
| Méthode | CC Index API → WARC Range requests → BeautifulSoup |
| Filtrage | Typosquatting FR, domaines bancaires, patterns d'arnaque |
| Validation | Langue (langdetect), longueur, dédup (SHA-256), mots-clés phishing |
| Output | `data/raw/common_crawl/` (CSV + rapport qualité JSON) |

### Enseignements clés (pour le rapport)

1. **Les pages de phishing sont éphémères** — les crawlers n'arrivent souvent pas à temps
2. **Le ratio signal/bruit est faible** pour le phishing web dans Common Crawl
3. **Les pages légitimes FR** (banques, gov) sont bien couvertes → utiles comme classe "ham"
4. **BigQuery reste la source big data principale** — Common Crawl est un complément exploratoire

**Compétences démontrées :**
- **C1** : Extraction depuis un système big data (Common Crawl, 250+ To)
- **C3** : Évaluation de la qualité des données avant traitement
- Esprit critique : documenter un résultat négatif est aussi valide qu'un positif